# 02: Fish Species and Part Identification

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, precision_recall_curve, average_precision_score
from fishy import TrainingConfig, run_unified_training, display_final_summary, create_data_module
pio.renderers.default = "notebook_connected"
try:
    import torch
    import torch.nn as nn
    from fishy.analysis.xai import GradCAM, ModelWrapper
    from lime.lime_tabular import LimeTabularExplainer
    from fishy._core.utils import get_device
    HAS_XAI = True
except ImportError:
    HAS_XAI = False
    print("XAI dependencies (lime, torch) not fully available.")


In [ ]:
results = run_unified_training(TrainingConfig(model="transformer", dataset="species", epochs=5, wandb_log=False))
display_final_summary(results)


## Performance & Interpretability

In [ ]:
if "predictions" in results:
    p = results["predictions"]; cn = results["class_names"]
    cm = confusion_matrix(p["labels"], p["preds"])
    px.imshow(cm, x=cn, y=cn, text_auto=True, title="Confusion Matrix", color_continuous_scale="Blues").show()


In [ ]:
if HAS_XAI and "model" in results and "data_module" in results:
    try:
        m = results["model"]; dm = results["data_module"]; X_x, y_x = dm.get_numpy_data(labels_as_indices=True)
        feature_names = [f"{c}" for c in dm.get_filtered_dataframe().columns if c not in ["Class Name", "m/z"]]
        explainer = LimeTabularExplainer(X_x, feature_names=feature_names, class_names=results["class_names"], discretize_continuous=True)
        exp = explainer.explain_instance(X_x[0], ModelWrapper(m, str(get_device())).predict_proba, num_features=10)
        el = exp.as_list()
        px.bar(x=[x[1] for x in el], y=[x[0] for x in el], orientation="h", title="LIME Explanation (Sample 0)").show()
    except Exception as e: print(f"XAI Visualization failed: {e}")
